## Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# For fuzzy matching company names
from fuzzywuzzy import fuzz, process

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

/opt/anaconda3/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


## Load Data

In [2]:
# Load funding rounds data
df_rounds = pd.read_csv('../data/raw/latestrounds_funding_rounds.csv')

print("Funding Rounds Data:")
print(f"  Rows: {len(df_rounds):,}")
print(f"  Columns: {list(df_rounds.columns)}")
print(f"\nFirst few rows:")
display(df_rounds.head())

print(f"\nUnique companies in rounds data: {df_rounds['company_name'].nunique():,}")

Funding Rounds Data:
  Rows: 1,063
  Columns: ['company_name', 'round_type', 'amount', 'date', 'investors']

First few rows:


,company_name,round_type,amount,date,investors
0,Graftcode,Seed,$2.1M,"Apr 14, 2026",SMOK Ventures
1,Helical,Seed,$10.0M,"Apr 14, 2026","AIX Ventures, Sequoia Scout Fund"
2,Modern Relay,Seed,$3.0M,"Apr 14, 2026","Moonfire Ventures, Bonsai Partners"
3,Obriy AI,Pre Seed,$500K,"Apr 14, 2026",angel investors
4,Stirlight,Seed,$1.6M,"Apr 15, 2026",NaN



Unique companies in rounds data: 496


In [3]:
# Load company-level data
df_companies = pd.read_csv('../data/raw/latestrounds_company_list.csv')

cols_to_drop = [
    'crunchbase_url',
    'latitude',
    'longitude',
    'valuation_usd',
    'num_founders',
    'founder_backgrounds',
    'serial_entrepreneur',
    'employee_count',
    'github_stars'
]

df_companies = df_companies.drop(columns=cols_to_drop)

print("Company-Level Data:")
print(f"  Rows: {len(df_companies):,}")
print(f"  Columns: {list(df_companies.columns)}")
print(f"\nFirst few rows:")
display(df_companies.head())

print(f"\nData types:")
print(df_companies.dtypes)

Company-Level Data:
  Rows: 502
  Columns: ['name', 'founded_date', 'status', 'headquarters_city', 'headquarters_country', 'primary_category', 'subcategory', 'total_funding_usd', 'last_funding_date', 'last_funding_type', 'num_investors', 'top_tier_investor', 'investor_names']

First few rows:


,name,founded_date,status,headquarters_city,headquarters_country,primary_category,subcategory,total_funding_usd,last_funding_date,last_funding_type,num_investors,top_tier_investor,investor_names
0,Graftcode,NaN,Active,"Warsaw, Poland",Funding Timeline,Artificial Intelligence,NaN,2100000.0,"Apr 14, 2026",Seed,1.0,N,SMOK Ventures
1,Helical,NaN,Active,"London, United Kingdom",Funding Timeline,Artificial Intelligence,NaN,10000000.0,"Apr 14, 2026",Seed,2.0,Y,"AIX Ventures, Sequoia Scout Fund"
2,Modern Relay,NaN,Active,"Barcelona, Spain",Funding Timeline,Artificial Intelligence,NaN,3000000.0,"Apr 14, 2026",Seed,2.0,N,"Moonfire Ventures, Bonsai Partners"
3,Obriy AI,NaN,Active,"Kyiv, Ukraine",Funding Timeline,Artificial Intelligence,NaN,500000.0,"Apr 14, 2026",Pre Seed,1.0,N,angel investors
4,Stirlight,NaN,Active,United Kingdom,Industry,Artificial Intelligence,"Industrial Tech, Manufacturing, quality-assurance",3300000.0,"Apr 15, 2026",Seed,1.0,N,Northern Gritstone



Data types:
name                     object
founded_date            float64
status                   object
headquarters_city        object
headquarters_country     object
primary_category         object
subcategory              object
total_funding_usd       float64
last_funding_date        object
last_funding_type        object
num_investors           float64
top_tier_investor        object
investor_names           object
dtype: object


## Step 1: Clean Company Names for Matching

In [4]:
def clean_company_name(name):
    """
    Standardize company names for matching.
    
    - Convert to lowercase
    - Remove common suffixes (Inc, Ltd, Corp, etc.)
    - Strip whitespace
    """
    if pd.isna(name):
        return name
    
    name = str(name).lower().strip()
    
    # Remove common company suffixes
    suffixes = ['inc.', 'inc', 'ltd.', 'ltd', 'corp.', 'corp', 'llc', 
                'corporation', 'limited', 'co.', 'company']
    for suffix in suffixes:
        if name.endswith(' ' + suffix):
            name = name[:-len(suffix)-1]
    
    return name.strip()

# Create cleaned name columns
df_rounds['company_name_clean'] = df_rounds['company_name'].apply(clean_company_name)
df_companies['name_clean'] = df_companies['name'].apply(clean_company_name)

print("Sample of cleaned names:")
print(df_companies[['name', 'name_clean']].head(10))

Sample of cleaned names:
             name      name_clean
0       Graftcode       graftcode
1         Helical         helical
2    Modern Relay    modern relay
3        Obriy AI        obriy ai
4       Stirlight       stirlight
5       TraqCheck       traqcheck
6           Ralio           ralio
7        Replenit        replenit
8  Round Treasury  round treasury
9           Haast           haast


## Step 2: Parse Funding Amounts

In [5]:
def parse_funding_amount(amount_str):
    """
    Convert funding string to numeric value in millions USD.
    
    Examples:
        '$5.2M' -> 5.2
        '$100K' -> 0.1
        '$1.5B' -> 1500.0
        '5.2M USD' -> 5.2
    """
    if pd.isna(amount_str):
        return np.nan
    
    # Convert to string and clean
    amount_str = str(amount_str).upper().replace(',', '').replace('$', '').replace('USD', '').strip()
    
    # Extract number and multiplier
    match = re.search(r'([\d.]+)\s*([MBK]?)', amount_str)
    
    if not match:
        return np.nan
    
    try:
        number = float(match.group(1))
        multiplier = match.group(2)
        
        if multiplier == 'B':
            return number * 1000  # Billions to millions
        elif multiplier == 'K':
            return number / 1000  # Thousands to millions
        else:  # M or no multiplier (assume millions)
            return number
    except:
        return np.nan

# Parse amounts in funding rounds
df_rounds['amount_millions'] = df_rounds['amount'].apply(parse_funding_amount)

# Show results
print(f"\nSuccessfully parsed: {df_rounds['amount_millions'].notna().sum():,} / {len(df_rounds):,} rounds")
print(f"\nSample of parsed amounts:")
display(df_rounds[['company_name', 'amount', 'amount_millions']].head(10))

# Statistics
print(f"\nFunding amount statistics (millions USD):")
print(df_rounds['amount_millions'].describe())


Successfully parsed: 1,006 / 1,063 rounds

Sample of parsed amounts:


,company_name,amount,amount_millions
0,Graftcode,$2.1M,2.1
1,Helical,$10.0M,10.0
2,Modern Relay,$3.0M,3.0
3,Obriy AI,$500K,0.5
4,Stirlight,$1.6M,1.6
5,Stirlight,$1.6M,1.6
6,TraqCheck,$8.0M,8.0
7,Ralio,$2.1M,2.1
8,Replenit,$2.5M,2.5
9,Round Treasury,$6.0M,6.0



Funding amount statistics (millions USD):
count      1006.000000
mean        384.430270
std        3976.439122
min           0.010000
25%           5.000000
50%          20.000000
75%          80.000000
max      110000.000000
Name: amount_millions, dtype: float64


In [6]:
# Parse total funding in company data

# Check if total_funding_usd column exists and what format it's in
if 'total_funding_usd' in df_companies.columns:
    print(f"\nSample of total_funding_usd values:")
    print(df_companies['total_funding_usd'].head(10))
    
    # If it's already numeric, just convert to millions
    if df_companies['total_funding_usd'].dtype in ['float64', 'int64']:
        df_companies['total_funding_millions'] = df_companies['total_funding_usd'] / 1_000_000
        print("\nTotal funding already numeric - converted to millions")
    else:
        # If it's string, parse it
        df_companies['total_funding_millions'] = df_companies['total_funding_usd'].apply(parse_funding_amount)
        print("\nTotal funding parsed from string")
    
    print(f"\nSuccessfully parsed: {df_companies['total_funding_millions'].notna().sum()} / {len(df_companies)} companies")
    print(f"\nTop 10 companies by total funding:")
    display(df_companies.nlargest(10, 'total_funding_millions')[['name', 'total_funding_usd', 'total_funding_millions']])


Sample of total_funding_usd values:
0     2100000.0
1    10000000.0
2     3000000.0
3      500000.0
4     3300000.0
5     8000000.0
6     2100000.0
7     2500000.0
8    10500000.0
9    17000000.0
Name: total_funding_usd, dtype: float64

Total funding already numeric - converted to millions

Successfully parsed: 479 / 502 companies

Top 10 companies by total funding:


,name,total_funding_usd,total_funding_millions
154,OpenAI,1.589000e+11,158900.0
141,xAI,8.880000e+10,88800.0
155,Anthropic,3.000000e+10,30000.0
329,Figma,2.110000e+10,21100.0
382,Stripe,7.900000e+09,7900.0
205,Anduril,6.700000e+09,6700.0
451,Stegra,3.100000e+09,3100.0
128,Skild AI,2.500000e+09,2500.0
375,PhonePe,2.100000e+09,2100.0
224,Nebius Group,2.000000e+09,2000.0


## Step 3: Split Location into City and Country

In [7]:
def standardize_country(country):
    """Normalize country labels for consistent analysis."""
    if pd.isna(country):
        return np.nan

    c = str(country).strip()
    if not c or c.lower() in {'nan', 'none'}:
        return np.nan

    upper = c.upper()
    if upper in {'US', 'USA', 'U.S.', 'U.S.A.', 'UNITED STATES', 'UNITED STATES OF AMERICA'}:
        return 'United States'
    if upper in {'UK', 'U.K.', 'UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND'}:
        return 'United Kingdom'

    # Remove obvious non-country artifacts
    if c.lower() in {'industry', 'funding timeline'}:
        return np.nan

    return c


# Build a country vocabulary from observed values + Babel territory names.
known_countries = set()
if 'headquarters_country' in df_companies.columns:
    known_countries = {
        str(x).strip() for x in df_companies['headquarters_country'].dropna().astype(str)
        if str(x).strip()
    }

try:
    from babel import Locale
    territory_names = {
        str(name).strip()
        for code, name in Locale('en').territories.items()
        if isinstance(code, str) and code.isalpha() and len(code) == 2
    }
    known_countries.update(territory_names)
except Exception:
    pass

# Common aliases not always present in territory names.
known_countries.update({
    'United States', 'United Kingdom', 'US', 'USA', 'U.S.', 'U.S.A.', 'UK', 'U.K.',
    'UAE', 'United Arab Emirates'
})


def is_probable_country_token(token):
    """Heuristic to determine if a standalone location token is a country name."""
    if pd.isna(token):
        return False

    t = str(token).strip()
    if not t or t.lower() in {'nan', 'none', 'industry', 'funding timeline'}:
        return False

    standardized = standardize_country(t)
    if pd.isna(standardized):
        return False

    return (t in known_countries) or (standardized in known_countries)


def split_location(location_str):
    """
    Split location string into city and country.

    Common formats:
        'San Francisco, CA, USA' -> ('San Francisco', 'USA')
        'London, UK' -> ('London', 'UK')
        'Tel Aviv, Israel' -> ('Tel Aviv', 'Israel')
        'United Kingdom' -> (None, 'United Kingdom')
    """
    if pd.isna(location_str):
        return None, None

    location_str = str(location_str).strip()
    if not location_str or location_str.lower() in {'nan', 'none'}:
        return None, None

    # Split by comma and drop empty tokens
    parts = [p.strip() for p in location_str.split(',') if p.strip()]

    if len(parts) == 0:
        return None, None
    elif len(parts) == 1:
        # If single token appears to be a country, keep city empty.
        token = parts[0]
        if is_probable_country_token(token):
            return None, token
        return token, None
    else:
        # City, Country OR City, State/Region, Country
        city = parts[0]
        country = parts[-1]

        # Guard against scraper metadata leaking into location field
        if country.lower() in {'industry', 'funding timeline'} and len(parts) >= 2:
            country = parts[-2]

        return city, country


# Apply to company data
if 'headquarters_city' in df_companies.columns:

    # Show sample of original locations
    print("\nSample of original locations:")
    print(df_companies['headquarters_city'].head(20))

    raw_location = df_companies['headquarters_city'].astype(str).str.strip()

    # Split location
    df_companies[['hq_city_parsed', 'hq_country_parsed']] = df_companies['headquarters_city'].apply(
        lambda x: pd.Series(split_location(x))
    )

    # Show results
    print("\nSample of parsed locations:")
    display(df_companies[['headquarters_city', 'hq_city_parsed', 'hq_country_parsed']].head(20))

    # Start with parsed city/country
    df_companies['headquarters_city_final'] = df_companies['hq_city_parsed']
    df_companies['headquarters_country_final'] = df_companies['hq_country_parsed']

    # Fallback country from dedicated country field if present
    if 'headquarters_country' in df_companies.columns:
        print("✓ Found existing headquarters_country column")
        df_companies['headquarters_country_final'] = df_companies['headquarters_country_final'].fillna(
            df_companies['headquarters_country']
        )

    # If raw location is country-only, keep city empty and use it as country fallback
    raw_is_country = raw_location.apply(is_probable_country_token)

    # Safe city fallback only when raw token is not a country
    city_fallback_mask = df_companies['headquarters_city_final'].isna() & (~raw_is_country)
    df_companies.loc[city_fallback_mask, 'headquarters_city_final'] = raw_location[city_fallback_mask]

    # Country fallback from raw location when token is country-like and country is still missing
    country_from_raw_mask = df_companies['headquarters_country_final'].isna() & raw_is_country
    df_companies.loc[country_from_raw_mask, 'headquarters_country_final'] = raw_location[country_from_raw_mask]

    # Normalize country values
    df_companies['headquarters_country_final'] = df_companies['headquarters_country_final'].apply(standardize_country)

    # Defensive cleanup: if city equals country, clear city.
    same_city_country = (
        df_companies['headquarters_city_final'].notna()
        & df_companies['headquarters_country_final'].notna()
        & (df_companies['headquarters_city_final'].astype(str).str.strip().str.lower()
           == df_companies['headquarters_country_final'].astype(str).str.strip().str.lower())
    )
    df_companies.loc[same_city_country, 'headquarters_city_final'] = np.nan

    print(f"\nFinal city coverage: {df_companies['headquarters_city_final'].notna().sum()} / {len(df_companies)} companies")
    print(f"Final country coverage: {df_companies['headquarters_country_final'].notna().sum()} / {len(df_companies)} companies")

    us_count = (df_companies['headquarters_country_final'] == 'United States').sum()
    print(f"Standardized 'United States' count: {us_count}")

else:
    print("No headquarters_city column found")
    # Use existing city/country if available
    if 'headquarters_city' in df_companies.columns:
        df_companies['headquarters_city_final'] = df_companies['headquarters_city']
    if 'headquarters_country' in df_companies.columns:
        df_companies['headquarters_country_final'] = df_companies['headquarters_country']

    if 'headquarters_country_final' in df_companies.columns:
        df_companies['headquarters_country_final'] = df_companies['headquarters_country_final'].apply(standardize_country)



Sample of original locations:
0                    Warsaw, Poland
1            London, United Kingdom
2                  Barcelona, Spain
3                     Kyiv, Ukraine
4                    United Kingdom
5                  Bengaluru, India
6            London, United Kingdom
7            London, United Kingdom
8            London, United Kingdom
9                 Sydney, Australia
10                  Langhus, Norway
11           Prague, Czech Republic
12     San Francisco, United States
13                 Bengaluru, India
14       Los Angeles, United States
15                    Paris, France
16    Milton Keynes, United Kingdom
17       Santa Clara, United States
18                     Milan, Italy
19                 Tampere, Finland
Name: headquarters_city, dtype: object

Sample of parsed locations:


,headquarters_city,hq_city_parsed,hq_country_parsed
0,"Warsaw, Poland",Warsaw,Poland
1,"London, United Kingdom",London,United Kingdom
2,"Barcelona, Spain",Barcelona,Spain
3,"Kyiv, Ukraine",Kyiv,Ukraine
4,United Kingdom,None,United Kingdom
5,"Bengaluru, India",Bengaluru,India
6,"London, United Kingdom",London,United Kingdom
7,"London, United Kingdom",London,United Kingdom
8,"London, United Kingdom",London,United Kingdom
9,"Sydney, Australia",Sydney,Australia


✓ Found existing headquarters_country column

Final city coverage: 461 / 502 companies
Final country coverage: 496 / 502 companies
Standardized 'United States' count: 203


## Step 4: Clean Dates

In [8]:
# Parse dates in funding rounds
df_rounds['date_parsed'] = pd.to_datetime(df_rounds['date'], errors='coerce')

print(f"Successfully parsed: {df_rounds['date_parsed'].notna().sum():,} / {len(df_rounds):,} rounds")
print(f"\nDate range: {df_rounds['date_parsed'].min()} to {df_rounds['date_parsed'].max()}")

# Parse founded date in company data
if 'founded_date' in df_companies.columns:
    founded_year_raw = pd.to_numeric(df_companies['founded_date'], errors='coerce')
    current_year = datetime.now().year
    df_companies['founded_year'] = founded_year_raw.where((founded_year_raw >= 1900) & (founded_year_raw <= current_year))
    df_companies['founded_year'] = df_companies['founded_year'].astype('Int64')
    df_companies['founded_date_parsed'] = pd.to_datetime(df_companies['founded_year'].astype('string'), format='%Y', errors='coerce')
    
    print(f"Successfully parsed: {df_companies['founded_date_parsed'].notna().sum()} / {len(df_companies)} companies")
    print(f"\nFounded year range: {df_companies['founded_year'].min()} to {df_companies['founded_year'].max()}")
else:
    print("\nNo founded_date column found")

# Parse last funding date in company data
if 'last_funding_date' in df_companies.columns:
    df_companies['last_funding_date_parsed'] = pd.to_datetime(df_companies['last_funding_date'], errors='coerce')
    print(f"Successfully parsed: {df_companies['last_funding_date_parsed'].notna().sum()} / {len(df_companies)} companies")

Successfully parsed: 1,063 / 1,063 rounds

Date range: 1995-01-01 00:00:00 to 2026-04-17 00:00:00
Successfully parsed: 384 / 502 companies

Founded year range: 1974 to 2026
Successfully parsed: 496 / 502 companies


## Step 5: Aggregate Funding Rounds by Company

In [9]:
# Aggregate rounds data
rounds_agg = df_rounds.groupby('company_name').agg({
    'amount_millions': ['sum', 'mean', 'max', 'count'],
    'date_parsed': ['max', 'min'],
    'round_type': lambda x: x.iloc[-1] if len(x) > 0 else None,
    'investors': lambda x: '; '.join(x.dropna().unique()[:10]) if len(x.dropna()) > 0 else None  # Top 10 unique
}).reset_index()

# Flatten column names
rounds_agg.columns = [
    'company_name',
    'rounds_total_funding_millions',  # Sum from rounds
    'rounds_avg_size_millions',
    'rounds_largest_millions',
    'num_rounds',
    'rounds_last_date',
    'rounds_first_date',
    'rounds_last_type',
    'rounds_investors'
]

print(f"\nAggregated {len(rounds_agg):,} companies from rounds data")
print(f"\nTop 10 by total funding (from rounds):")
display(rounds_agg.nlargest(10, 'rounds_total_funding_millions')[[
    'company_name', 'rounds_total_funding_millions', 'num_rounds', 'rounds_largest_millions'
]])


Aggregated 496 companies from rounds data

Top 10 by total funding (from rounds):


,company_name,rounds_total_funding_millions,num_rounds,rounds_largest_millions
285,OpenAI,158900.0,6,110000.0
495,xAI,88769.7,13,20000.0
26,Anthropic,30000.0,1,30000.0
147,Figma,21052.9,12,20000.0
406,Stripe,7940.0,9,6900.0
24,Anduril,6697.0,11,2500.0
401,Stegra,3100.0,2,1700.0
385,Skild AI,2514.5,5,1400.0
310,PhonePe,2072.0,8,1500.0
256,Nebius Group,2000.0,1,2000.0


## Step 6: Join Datasets

In [10]:
# Create clean names for joining
rounds_agg['company_name_clean'] = rounds_agg['company_name'].apply(clean_company_name)

# Join on cleaned names
df_merged = df_companies.merge(
    rounds_agg,
    left_on='name_clean',
    right_on='company_name_clean',
    how='outer',  # Keep all companies from both datasets
    indicator=True,  # Track which dataset each row came from
    suffixes=('_company', '_rounds')
)

print(f"\nMerge results:")
print(df_merged['_merge'].value_counts())
print(f"\nTotal companies in merged dataset: {len(df_merged):,}")

# Use company name from company data if available, else from rounds
df_merged['company_name_final'] = df_merged['name'].fillna(df_merged['company_name'])

print(f"\nSample of merged data:")
display(df_merged[[
    'company_name_final', '_merge', 'total_funding_millions', 'rounds_total_funding_millions', 'num_rounds'
]].head(10))


Merge results:
_merge
both          496
left_only       6
right_only      0
Name: count, dtype: int64

Total companies in merged dataset: 502

Sample of merged data:


,company_name_final,_merge,total_funding_millions,rounds_total_funding_millions,num_rounds
0,1Buy.ai,both,3.0,3.00,1.0
1,360 ONE Asset,both,109.8,109.80,1.0
2,4baseCare,both,8.0,8.00,2.0
3,7AI,both,130.0,130.00,1.0
4,9fin,both,159.8,159.80,1.0
5,Accelsius,both,89.3,89.25,3.0
6,Adaptive6,both,44.0,44.00,2.0
7,Adonis,both,40.0,40.00,1.0
8,Aerem,both,15.0,15.00,1.0
9,Agaton,both,10.0,10.00,1.0


## Step 7: Reconcile Total Funding

We have two sources of total funding. Let's compare and choose the best value.

In [11]:
# Compare the two funding sources

# For companies in both datasets
both = df_merged[df_merged['_merge'] == 'both'].copy()

if len(both) > 0:
    # Calculate difference
    both['funding_diff'] = both['total_funding_millions'] - both['rounds_total_funding_millions']
    both['funding_diff_pct'] = (both['funding_diff'] / both['total_funding_millions'] * 100).abs()
    
    print(f"\nCompanies in both datasets: {len(both)}")
    print(f"\nFunding comparison statistics:")
    print(both[['total_funding_millions', 'rounds_total_funding_millions', 'funding_diff', 'funding_diff_pct']].describe())
    
    # Show cases with large discrepancies
    large_diff = both[both['funding_diff_pct'] > 20].nlargest(10, 'funding_diff_pct')
    if len(large_diff) > 0:
        print(f"\n⚠️ Companies with >20% funding discrepancy:")
        display(large_diff[[
            'company_name_final', 'total_funding_millions', 'rounds_total_funding_millions', 
            'funding_diff', 'funding_diff_pct'
        ]])

# Create final total funding column (use company data if available, else use rounds sum)
df_merged['total_funding_final'] = df_merged['total_funding_millions'].fillna(
    df_merged['rounds_total_funding_millions']
)

print(f"\nFinal total funding coverage: {df_merged['total_funding_final'].notna().sum()} / {len(df_merged)} companies")


Companies in both datasets: 496

Funding comparison statistics:
       total_funding_millions  rounds_total_funding_millions  funding_diff  \
count              479.000000                     496.000000    479.000000   
mean               807.509626                     779.711395      0.125802   
std               8476.308909                    8329.977107      6.204842   
min                  0.010000                       0.000000    -44.500000   
25%                  5.100000                       4.075000      0.000000   
50%                 26.000000                      23.000000      0.000000   
75%                130.950000                     125.500000      0.000000   
max             158900.000000                  158900.000000     50.000000   

       funding_diff_pct  
count        479.000000  
mean           0.081405  
std            0.442041  
min            0.000000  
25%            0.000000  
50%            0.000000  
75%            0.000000  
max            4.545455 

## Step 8: Create Derived Variables for Regression

In [12]:
# 1. Company age
current_year = datetime.now().year
df_merged['company_age_years'] = current_year - df_merged['founded_year']

print(f"1. Company age:")
print(f"   Companies with age data: {df_merged['company_age_years'].notna().sum()}")
if df_merged['company_age_years'].notna().sum() > 0:
    print(f"   Age range: {df_merged['company_age_years'].min():.0f} to {df_merged['company_age_years'].max():.0f} years")

# Ensure country labels are standardized after merge too
if 'headquarters_country_final' in df_merged.columns:
    df_merged['headquarters_country_final'] = df_merged['headquarters_country_final'].apply(standardize_country)

# 2. Location dummies
print(f"\n2. Location variables:")

# Bay Area cities
bay_area_cities = ['San Francisco', 'San Jose', 'Palo Alto', 'Mountain View',
                   'Menlo Park', 'Sunnyvale', 'Santa Clara', 'Redwood City',
                   'Cupertino', 'Fremont', 'Oakland', 'Berkeley']

df_merged['is_bay_area'] = df_merged['headquarters_city_final'].isin(bay_area_cities).astype(int)
print(f"   Bay Area companies: {df_merged['is_bay_area'].sum()}")

# USA dummy
df_merged['is_usa'] = (
    df_merged['headquarters_country_final'] == 'United States'
).astype(int)
print(f"   USA companies: {df_merged['is_usa'].sum()}")

# Other key locations
df_merged['is_nyc'] = df_merged['headquarters_city_final'].isin(['New York', 'NYC', 'New York City']).astype(int)
df_merged['is_london'] = df_merged['headquarters_city_final'].str.contains('London', case=False, na=False).astype(int)
df_merged['is_telaviv'] = df_merged['headquarters_city_final'].str.contains('Tel Aviv', case=False, na=False).astype(int)

# 3. Subsector/Category dummies
print(f"\n3. Subsector variables:")

category_cols = [c for c in ['subcategory', 'primary_category', 'subsector'] if c in df_merged.columns]
if category_cols:
    print(f"   Using columns: {', '.join(category_cols)}")

    category_text = (
        df_merged[category_cols]
        .fillna('')
        .astype(str)
        .agg(' | '.join, axis=1)
    )

    has_category_data = category_text.str.strip().ne('')
    print(f"   Companies with category data: {has_category_data.sum()}")

    pattern_map = {
        # Technical AI buckets
        'is_llm': r'\b(?:llm|large language model|foundation model|generative ai|gpt|language model)\b',
        'is_computer_vision': r'\b(?:computer vision|vision ai|image ai|image analysis|video ai|video generation|ocr|synthetic media)\b',
        'is_robotics': r'\b(?:robotics?|autonomous systems?|drone tech|human-robot interaction|self-driving|robotics-as-a-service)\b',
        'is_mlops': r'\b(?:mlops|devops|ai infrastructure|infrastructure software|cloud infrastructure|data engineering|developer tools|platform|api|database|databases|ai-infra)\b',
        'is_agent': r'\b(?:agent|agents|workflow automation|automation|rpa|conversational ai|voice ai|assistant)\b',

        # Vertical AI buckets
        'is_fin_ai': r'\b(?:fintech|payments?|banking|financial services|wealth|insurtech|regtech|lending)\b',
        'is_health_ai': r'\b(?:healthtech|digital health|diagnostics|medical|medtech|biotech|biotechnology|healthcare|therapeutics)\b',
        'is_security_ai': r'\b(?:cybersecurity|cloud security|data security|security software|threat intelligence|security)\b',
        'is_climate_ai': r'\b(?:climate tech|climatetech|clean tech|clean technology|cleantech|sustainability|energy tech|energy|carbon)\b',
        'is_enterprise_ai': r'\b(?:enterprise software|enterprise ai|enterprise saas|b2b saas|saas|workflow|productivity)\b',
    }

    for flag_col, pattern in pattern_map.items():
        df_merged[flag_col] = category_text.str.contains(pattern, case=False, na=False, regex=True).astype(int)

    # Catch-all bucket for category text that did not hit any explicit bucket
    mapped_flags = list(pattern_map.keys())
    df_merged['is_other_ai'] = ((has_category_data) & (df_merged[mapped_flags].sum(axis=1) == 0)).astype(int)

    print(f"   LLM companies: {df_merged['is_llm'].sum()}")
    print(f"   Computer Vision companies: {df_merged['is_computer_vision'].sum()}")
    print(f"   Robotics companies: {df_merged['is_robotics'].sum()}")
    print(f"   MLOps companies: {df_merged['is_mlops'].sum()}")
    print(f"   Agent companies: {df_merged['is_agent'].sum()}")
    print(f"   Fin AI companies: {df_merged['is_fin_ai'].sum()}")
    print(f"   Health AI companies: {df_merged['is_health_ai'].sum()}")
    print(f"   Security AI companies: {df_merged['is_security_ai'].sum()}")
    print(f"   Climate AI companies: {df_merged['is_climate_ai'].sum()}")
    print(f"   Enterprise AI companies: {df_merged['is_enterprise_ai'].sum()}")
    print(f"   Other AI companies: {df_merged['is_other_ai'].sum()}")
else:
    print(f"   No category data found")
    for var in [
        'is_llm', 'is_computer_vision', 'is_robotics', 'is_mlops', 'is_agent',
        'is_fin_ai', 'is_health_ai', 'is_security_ai', 'is_climate_ai', 'is_enterprise_ai', 'is_other_ai'
    ]:
        df_merged[var] = 0

# 4. Funding milestones
print(f"\n4. Funding milestone variables:")

df_merged['has_mega_round'] = (df_merged['rounds_largest_millions'] >= 50).astype(int)
df_merged['is_unicorn'] = (df_merged['total_funding_final'] >= 1000).astype(int)  # $1B+

print(f"   Companies with mega-round ($50M+): {df_merged['has_mega_round'].sum()}")
print(f"   Unicorns ($1B+ funding): {df_merged['is_unicorn'].sum()}")

# 5. Funding velocity (millions per year)
df_merged['funding_velocity'] = (
    df_merged['total_funding_final'] /
    df_merged['company_age_years'].replace(0, np.nan)
)

print(f"\n5. Other derived variables:")
print(f"   Companies with funding velocity: {df_merged['funding_velocity'].notna().sum()}")

# 6. Log transformation for regression
df_merged['log_total_funding'] = np.log(df_merged['total_funding_final'].replace(0, np.nan))
print(f"   Companies with log_total_funding: {df_merged['log_total_funding'].notna().sum()}")


1. Company age:
   Companies with age data: 384
   Age range: 0 to 52 years

2. Location variables:
   Bay Area companies: 93
   USA companies: 203

3. Subsector variables:
   Using columns: subcategory, primary_category
   Companies with category data: 502
   LLM companies: 6
   Computer Vision companies: 1
   Robotics companies: 13
   MLOps companies: 54
   Agent companies: 26
   Fin AI companies: 90
   Health AI companies: 63
   Security AI companies: 27
   Climate AI companies: 37
   Enterprise AI companies: 98
   Other AI companies: 153

4. Funding milestone variables:
   Companies with mega-round ($50M+): 175
   Unicorns ($1B+ funding): 25

5. Other derived variables:
   Companies with funding velocity: 380
   Companies with log_total_funding: 479


## Step 9: Handle Missing Values

In [13]:
print("Missing value analysis:")
print("=" * 70)

# Key variables for regression
key_vars = [
    'company_name_final',
    'total_funding_final',
    'headquarters_city_final',
    'headquarters_country_final',
    'founded_year',
    'company_age_years',
    'num_rounds',
    'is_bay_area',
    'is_usa',
    'is_llm',
    'is_computer_vision',
    'is_robotics'
]

missing = df_merged[key_vars].isnull().sum()
missing_pct = (missing / len(df_merged) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Percentage', ascending=False)

print(missing_df)

# Identify companies with complete data for regression
regression_vars = ['total_funding_final', 'company_age_years', 'is_bay_area']
complete_mask = df_merged[regression_vars].notna().all(axis=1)

print(f"\nCompanies with complete core data: {complete_mask.sum()} / {len(df_merged)} ({complete_mask.sum()/len(df_merged)*100:.1f}%)")

Missing value analysis:
                            Missing Count  Percentage
founded_year                          118       23.51
company_age_years                     118       23.51
headquarters_city_final                41        8.17
total_funding_final                     6        1.20
headquarters_country_final              6        1.20
num_rounds                              6        1.20
company_name_final                      0        0.00
is_bay_area                             0        0.00
is_usa                                  0        0.00
is_llm                                  0        0.00
is_computer_vision                      0        0.00
is_robotics                             0        0.00

Companies with complete core data: 384 / 502 (76.5%)


## Step 10: Final Dataset Selection

In [14]:
# Select final columns for analysis
final_columns = [
    # Identifiers
    'company_name_final',

    # Core characteristics
    'headquarters_city_final',
    'headquarters_country_final',
    'founded_year',
    'company_age_years',

    # Funding variables
    'total_funding_final',
    'log_total_funding',
    'num_rounds',
    'rounds_largest_millions',
    'funding_velocity',

    # Dates
    'founded_date_parsed',
    'last_funding_date_parsed',

    # Categories
    'primary_category' if 'primary_category' in df_merged.columns else None,
    'subcategory' if 'subcategory' in df_merged.columns else None,

    # Location dummies
    'is_bay_area',
    'is_usa',
    'is_nyc',
    'is_london',
    'is_telaviv',

    # Subsector dummies (technical)
    'is_llm',
    'is_computer_vision',
    'is_robotics',
    'is_mlops',
    'is_agent',

    # Subsector dummies (vertical + catch-all)
    'is_fin_ai' if 'is_fin_ai' in df_merged.columns else None,
    'is_health_ai' if 'is_health_ai' in df_merged.columns else None,
    'is_security_ai' if 'is_security_ai' in df_merged.columns else None,
    'is_climate_ai' if 'is_climate_ai' in df_merged.columns else None,
    'is_enterprise_ai' if 'is_enterprise_ai' in df_merged.columns else None,
    'is_other_ai' if 'is_other_ai' in df_merged.columns else None,

    # Milestones
    'has_mega_round',
    'is_unicorn',

    # Additional info
    'employee_count' if 'employee_count' in df_merged.columns else None,
    'num_founders' if 'num_founders' in df_merged.columns else None,
    'serial_entrepreneur' if 'serial_entrepreneur' in df_merged.columns else None,
    'top_tier_investor' if 'top_tier_investor' in df_merged.columns else None,
]

# Remove None values
final_columns = [col for col in final_columns if col is not None]

# Create final dataset
df_final = df_merged[final_columns].copy()

# Rename for clarity
df_final = df_final.rename(columns={
    'company_name_final': 'company_name',
    'headquarters_city_final': 'headquarters_city',
    'headquarters_country_final': 'headquarters_country',
    'total_funding_final': 'total_funding_millions',
    'founded_date_parsed': 'founded_date',
    'last_funding_date_parsed': 'last_funding_date'
})

print(f"Final dataset shape: {df_final.shape}")
print(f"\nColumns in final dataset:")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i}. {col}")


Final dataset shape: (502, 33)

Columns in final dataset:
  1. company_name
  2. headquarters_city
  3. headquarters_country
  4. founded_year
  5. company_age_years
  6. total_funding_millions
  7. log_total_funding
  8. num_rounds
  9. rounds_largest_millions
  10. funding_velocity
  11. founded_date
  12. last_funding_date
  13. primary_category
  14. subcategory
  15. is_bay_area
  16. is_usa
  17. is_nyc
  18. is_london
  19. is_telaviv
  20. is_llm
  21. is_computer_vision
  22. is_robotics
  23. is_mlops
  24. is_agent
  25. is_fin_ai
  26. is_health_ai
  27. is_security_ai
  28. is_climate_ai
  29. is_enterprise_ai
  30. is_other_ai
  31. has_mega_round
  32. is_unicorn
  33. top_tier_investor


## Step 11: Data Quality Summary

In [15]:
print("=" * 70)
print("FINAL DATASET QUALITY SUMMARY")
print("=" * 70)

print(f"\nTotal companies: {len(df_final):,}")

# Completeness by key variable
print(f"\nData Completeness:")
completeness = {
    'Total Funding': df_final['total_funding_millions'].notna().sum(),
    'Location (City)': df_final['headquarters_city'].notna().sum(),
    'Location (Country)': df_final['headquarters_country'].notna().sum(),
    'Founded Year': df_final['founded_year'].notna().sum(),
    'Number of Rounds': df_final['num_rounds'].notna().sum(),
    'Category/Subsector': (df_final['primary_category'].notna().sum() if 'primary_category' in df_final.columns else 0),
}

for var, count in completeness.items():
    pct = count / len(df_final) * 100
    print(f"  {var}: {count:,} ({pct:.1f}%)")

# Distribution summaries
print(f"\nFunding Statistics (millions USD):")
print(df_final['total_funding_millions'].describe())

print(f"\nTop 10 Companies by Total Funding:")
display(df_final.nlargest(10, 'total_funding_millions')[[
    'company_name', 'total_funding_millions', 'num_rounds',
    'headquarters_city', 'headquarters_country', 'founded_year'
]])

print(f"\nGeographic Distribution (Top 10 Countries):")
print(df_final['headquarters_country'].value_counts().head(10))

print(f"\nSubsector Distribution:")
subsector_counts = {
    'LLM': df_final['is_llm'].sum(),
    'Computer Vision': df_final['is_computer_vision'].sum(),
    'Robotics': df_final['is_robotics'].sum(),
    'MLOps': df_final['is_mlops'].sum(),
    'Agents': df_final['is_agent'].sum(),
    'Fin AI': (df_final['is_fin_ai'].sum() if 'is_fin_ai' in df_final.columns else 0),
    'Health AI': (df_final['is_health_ai'].sum() if 'is_health_ai' in df_final.columns else 0),
    'Security AI': (df_final['is_security_ai'].sum() if 'is_security_ai' in df_final.columns else 0),
    'Climate AI': (df_final['is_climate_ai'].sum() if 'is_climate_ai' in df_final.columns else 0),
    'Enterprise AI': (df_final['is_enterprise_ai'].sum() if 'is_enterprise_ai' in df_final.columns else 0),
    'Other AI': (df_final['is_other_ai'].sum() if 'is_other_ai' in df_final.columns else 0),
}
for sector, count in subsector_counts.items():
    print(f"  {sector}: {count}")


FINAL DATASET QUALITY SUMMARY

Total companies: 502

Data Completeness:
  Total Funding: 496 (98.8%)
  Location (City): 461 (91.8%)
  Location (Country): 496 (98.8%)
  Founded Year: 384 (76.5%)
  Number of Rounds: 496 (98.8%)
  Category/Subsector: 502 (100.0%)

Funding Statistics (millions USD):
count       496.000000
mean        779.832885
std        8330.782611
min           0.000000
25%           4.075000
50%          23.000000
75%         125.500000
max      158900.000000
Name: total_funding_millions, dtype: float64

Top 10 Companies by Total Funding:


,company_name,total_funding_millions,num_rounds,headquarters_city,headquarters_country,founded_year
295,OpenAI,158900.0,6.0,San Francisco,United States,2015
490,xAI,88800.0,13.0,San Francisco,United States,2023
25,Anthropic,30000.0,1.0,San Francisco,United States,2021
152,Figma,21100.0,12.0,San Francisco,United States,2012
417,Stripe,7900.0,9.0,Seattle,United States,2010
23,Anduril,6700.0,11.0,Costa Mesa,United States,2017
412,Stegra,3100.0,2.0,Stockholm,Sweden,2020
395,Skild AI,2500.0,5.0,Pittsburgh,United States,2023
321,PhonePe,2100.0,8.0,Bangalore,India,2015
264,Nebius Group,2000.0,1.0,Amsterdam,Netherlands,2023



Geographic Distribution (Top 10 Countries):
headquarters_country
United States     203
India              77
United Kingdom     55
Germany            17
France             14
Finland            13
Sweden             12
Netherlands        11
Spain              10
Singapore          10
Name: count, dtype: int64

Subsector Distribution:
  LLM: 6
  Computer Vision: 1
  Robotics: 13
  MLOps: 54
  Agents: 26
  Fin AI: 90
  Health AI: 63
  Security AI: 27
  Climate AI: 37
  Enterprise AI: 98
  Other AI: 153


## Step 12: Save Cleaned Data

In [16]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save final cleaned dataset
output_path = '../data/processed/ai_companies_cleaned.csv'
df_final.to_csv(output_path, index=False)

print(f"✓ Saved cleaned dataset: {output_path}")
print(f"  Rows: {len(df_final):,}")
print(f"  Columns: {len(df_final.columns)}")

# Also save the full merged dataset (for reference)
full_output_path = '../data/processed/ai_companies_full_merged.csv'
df_merged.to_csv(full_output_path, index=False)
print(f"\n✓ Saved full merged dataset: {full_output_path}")

# Save summary statistics
summary_stats = df_final.describe(include='all').T
summary_stats.to_csv('../data/processed/summary_statistics.csv')
print(f"\n✓ Saved summary statistics: ../data/processed/summary_statistics.csv")

✓ Saved cleaned dataset: ../data/processed/ai_companies_cleaned.csv
  Rows: 502
  Columns: 33

✓ Saved full merged dataset: ../data/processed/ai_companies_full_merged.csv

✓ Saved summary statistics: ../data/processed/summary_statistics.csv
